# TruthShield 01: Feature caching (run once)
Runtime → Change runtime type → **T4 GPU**. Needs notebook 00 to have run once (dataset, weights and splits on Drive).

Goes through all 3,810 HiFi-AVDF videos **once** and saves, per video:
- the HAVIC feature and score,
- 8 face crops for the frequency branch,
- face and voice emotion embeddings,
- 4 quality numbers.

**Resumable:** each video is saved to Drive as soon as it is done. After a disconnect, run all again and it continues where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, zipfile
ROOT = '/content/drive/MyDrive/TruthShield'
OUT = f'{ROOT}/cache/hifi'

%cd /content
if not os.path.exists('/content/HAVIC'):
    !git clone -q --depth 1 https://github.com/tuffy-studio/HAVIC /content/HAVIC
if not os.path.exists('/content/TruthShield'):
    !git clone -q https://github.com/Jawad374/TruthShield.git /content/TruthShield
!git -C /content/TruthShield pull -q
!pip install -q decord ffmpeg-python einops timm transformers
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'   # FaceX-Zoo pickled models
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## Videos to local disk (skipped if already there this session)

In [ ]:
LOCAL = '/content/hifi'
os.makedirs(LOCAL, exist_ok=True)
for z in sorted(glob.glob(f'{ROOT}/data/HiFi-AVDF/*.zip')):
    eng = os.path.basename(z)[:-4]
    if not os.path.exists(f'{LOCAL}/.done_{eng}'):
        print('unzipping', eng)
        with zipfile.ZipFile(z) as f:
            f.extractall(LOCAL)
        open(f'{LOCAL}/.done_{eng}', 'w').close()
SPLIT = f'{ROOT}/splits/hifi_main_abs.csv'
import pandas as pd
s = pd.read_csv(SPLIT)
print(len(s), 'clips in split |', sum(os.path.exists(p) for p in s.abs_path), 'found on local disk')

## 1. Smoke test: 5 videos
Check that each line shows `havic_p=...` and not `FAILED`, and note the seconds per video.
These 5 are saved and skipped later, so nothing is wasted.

In [ ]:
%cd /content/TruthShield
!python -u cache_features.py --split_csv {SPLIT} --havic_root /content/HAVIC --havic_weights {ROOT}/weights/best_ft_model_clean.pth --out_dir {OUT} --limit 5 --debug

## 2. Full run (all 3,810 videos)
Prints one line per video with time left. Keep the tab open and click in it now and then.
If Colab disconnects, reconnect and **Run all**: finished videos are skipped.

In [ ]:
%cd /content/TruthShield
!python -u cache_features.py --split_csv {SPLIT} --havic_root /content/HAVIC --havic_weights {ROOT}/weights/best_ft_model_clean.pth --out_dir {OUT}

## 3. Check and pack into 2 files for fast training

In [ ]:
n = len(glob.glob(f'{OUT}/clips/*.pt'))
print(f'{n} / {len(s)} clips cached')
if os.path.exists(f'{OUT}/failures.csv'):
    f = pd.read_csv(f'{OUT}/failures.csv').drop_duplicates('video_path')
    f = f[~f.video_path.map(lambda p: os.path.exists(f"{OUT}/clips/{p.replace('/', '__').replace('.mp4', '.pt')}"))]
    print(len(f), 'clips failed:'); print(f.error.str.slice(0, 60).value_counts().head(10))

In [ ]:
%cd /content/TruthShield
!python -u cache_features.py --pack --out_dir {OUT}
!ls -lh {OUT}/features.pt {OUT}/faces.npy